# CW-006 Zero-Cost Colab Qualification

This runner qualifies PR #482 at exact code head `c61370664488e4ad3e274ec61a233c46652ec6de` without changing the PR branch.

Constraints:
- Google Colab **free CPU runtime only**.
- No PHI or patient audio.
- One public Google FLEURS validation sample (CC BY 4.0).
- Model: `openai/whisper-large-v3-turbo` at pinned revision `41f01f3fe87f28c78e2fbf8b568835947dd65ed9`.
- Network is allowed only for acquisition; backend construction and inference run after an explicit network-denial guard.
- No accuracy/WER/clinical-quality claim is made.
- Any payment request means STOP.

In [ ]:
import os, sys, subprocess
print("python", sys.version)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.13.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.16.1", "datasets==4.1.1", "soundfile==0.13.1"], check=True)
print("DEPENDENCIES_INSTALLED")

In [ ]:
import json, platform, hashlib, pathlib, subprocess, sys, os
import torch, transformers
runtime_meta = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "colab": "google.colab" in sys.modules,
    "gpu_available": bool(torch.cuda.is_available()),
}
assert transformers.__version__ == "5.16.1"
assert torch.__version__.split("+")[0] == "2.13.0"
print(json.dumps(runtime_meta, indent=2))

In [ ]:
from pathlib import Path
REPO = Path("/content/MESC")
if REPO.exists():
    subprocess.run(["rm", "-rf", str(REPO)], check=True)
subprocess.run(["git", "clone", "-q", "https://github.com/TheHalfMoon/MESC.git", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "c61370664488e4ad3e274ec61a233c46652ec6de"], check=True)
actual_head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert actual_head == "c61370664488e4ad3e274ec61a233c46652ec6de", actual_head
sys.path.insert(0, str(REPO / "apps" / "workspace" / "src"))
from medscale_workspace import asr as asr_mod
manifest = asr_mod.expected_manifest()
assert manifest.model_revision == "41f01f3fe87f28c78e2fbf8b568835947dd65ed9"
print("EXACT_HEAD", actual_head)
print("CONTRACT", asr_mod.adapter_contract_version())

In [ ]:
from huggingface_hub import HfApi, snapshot_download
from datasets import load_dataset, Audio
import shutil, json, os

MODEL_DIR = Path("/content/cw006-model")
if MODEL_DIR.exists():
    shutil.rmtree(MODEL_DIR)

dataset_info = HfApi().dataset_info("google/fleurs")
card = getattr(dataset_info, "card_data", None)
dataset_license = getattr(card, "license", None)
if dataset_license is None and hasattr(card, "to_dict"):
    dataset_license = card.to_dict().get("license")
if isinstance(dataset_license, (list, tuple, set)):
    normalized_licenses = {str(item).lower() for item in dataset_license}
else:
    normalized_licenses = {str(dataset_license).lower()}

assert "cc-by-4.0" in normalized_licenses, dataset_license
dataset_license = "cc-by-4.0"

snapshot_path = snapshot_download(
    repo_id=asr_mod.MODEL_ID,
    revision=asr_mod.MODEL_REVISION,
    local_dir=str(MODEL_DIR),
)
snapshot_path = Path(snapshot_path)
asr_mod.verify_local_snapshot(snapshot_path, manifest)

fleurs = load_dataset("google/fleurs", "en_us", split="validation[:1]")
fleurs = fleurs.cast_column("audio", Audio(decode=False))
sample = fleurs[0]
audio_record = sample["audio"]
assert isinstance(audio_record, dict)
source_bytes = audio_record.get("bytes")
source_path = audio_record.get("path")
if source_bytes is None and source_path and Path(source_path).is_file():
    source_bytes = Path(source_path).read_bytes()
assert source_bytes is not None, audio_record
sample_meta = {
    "dataset": "google/fleurs",
    "config": "en_us",
    "split": "validation[:1]",
    "license": str(dataset_license),
    "id": str(sample.get("id", "")),
    "source_audio_sha256": hashlib.sha256(source_bytes).hexdigest(),
    "reference_transcription": str(sample.get("transcription", "")),
}
print(json.dumps({k:v for k,v in sample_meta.items() if k != "reference_transcription"}, indent=2))

In [ ]:
import io, wave, numpy as np, soundfile as sf
data, sr = sf.read(io.BytesIO(source_bytes), dtype="float32", always_2d=False)
if data.ndim == 2:
    data = data.mean(axis=1)
data = np.asarray(data, dtype=np.float32)
if int(sr) != asr_mod.SAMPLES_PER_SECOND:
    old_x = np.arange(len(data), dtype=np.float64) / float(sr)
    new_len = max(1, int(round(len(data) * asr_mod.SAMPLES_PER_SECOND / float(sr))))
    new_x = np.arange(new_len, dtype=np.float64) / float(asr_mod.SAMPLES_PER_SECOND)
    data = np.interp(new_x, old_x, data).astype(np.float32)
threshold_hits = np.flatnonzero(np.abs(data) > 0.01)
speech_start = int(threshold_hits[0]) if len(threshold_hits) else 0
crop_start = max(0, speech_start - 1600)
max_frames = min(32000, (asr_mod.MAXIMUM_AUDIO_BYTES - 128) // 2)
clip = data[crop_start:crop_start + max_frames]
assert len(clip) > 0
pcm = (np.clip(clip, -1.0, 1.0) * 32767.0).astype("<i2")
buf = io.BytesIO()
with wave.open(buf, "wb") as w:
    w.setnchannels(1)
    w.setsampwidth(2)
    w.setframerate(asr_mod.SAMPLES_PER_SECOND)
    w.writeframes(pcm.tobytes())
audio_bytes = buf.getvalue()
assert len(audio_bytes) <= asr_mod.MAXIMUM_AUDIO_BYTES, (len(audio_bytes), asr_mod.MAXIMUM_AUDIO_BYTES)
sample_meta.update({
    "source_sample_rate": int(sr),
    "derived_clip_start_sample_16k": crop_start,
    "derived_clip_frames_16k": int(len(clip)),
    "derived_wav_bytes": len(audio_bytes),
    "derived_wav_sha256": hashlib.sha256(audio_bytes).hexdigest(),
})
print(json.dumps({k:v for k,v in sample_meta.items() if k != "reference_transcription"}, indent=2))

In [ ]:
from medscale_workspace.errors import AsrModelUnavailableError
missing_snapshot_pass = False
try:
    asr_mod.verify_local_snapshot("/content/cw006-definitely-missing", manifest)
except AsrModelUnavailableError:
    missing_snapshot_pass = True
assert missing_snapshot_pass

import socket
network_attempts = []
_original_connect = socket.socket.connect
_original_create_connection = socket.create_connection

def _deny_connect(self, address):
    network_attempts.append({"kind":"socket.connect","address":repr(address)})
    raise RuntimeError("CW006 network denied after acquisition")

def _deny_create_connection(address, *args, **kwargs):
    network_attempts.append({"kind":"socket.create_connection","address":repr(address)})
    raise RuntimeError("CW006 network denied after acquisition")

os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
socket.socket.connect = _deny_connect
socket.create_connection = _deny_create_connection
print("NETWORK_DENIAL_ACTIVE")

In [ ]:
from uuid import UUID
backend = asr_mod.TransformersWhisperBackend(snapshot_path, manifest)
result = asr_mod.transcribe_with_backend(
    UUID("6cd9e9f4-1c3b-40b4-b10f-5b16db71a9fe"),
    UUID("9c0d1e2f-3a4b-4c5d-8e6f-7a8b9c0d1e2f"),
    UUID("1b2c3d4e-5f6a-7b8c-9d0e-1f2a3b4c5d6e"),
    "chunk-00000001",
    audio_bytes,
    "en",
    "2026-09-23T13:00:00+00:00",
    "2026-09-23T13:01:00+00:00",
    manifest,
    True,
    asr_mod.MODEL_REVISION,
    False,
    True,
    False,
    backend,
)
assert result.status.value == "success", result
assert result.transcript.strip() != ""
assert network_attempts == [], network_attempts
print("REAL_BACKEND_STATUS", result.status.value)
print("REAL_BACKEND_TRANSCRIPT", result.transcript)

In [ ]:
evidence = {
    "qualification": "CW-006 real-model runtime-path qualification",
    "repo": "TheHalfMoon/MESC",
    "pr": 482,
    "exact_head": actual_head,
    "runtime": runtime_meta,
    "model": {
        "id": asr_mod.MODEL_ID,
        "revision": asr_mod.MODEL_REVISION,
        "license": asr_mod.MODEL_LICENSE,
        "weight_sha256": asr_mod.WEIGHT_SHA256,
        "weight_size_bytes": asr_mod.WEIGHT_SIZE_BYTES,
        "snapshot_verified": True,
        "local_files_only": True,
        "trust_remote_code": False,
        "allow_download": False,
    },
    "fixture": sample_meta,
    "missing_snapshot_fail_closed": missing_snapshot_pass,
    "network_denial_active_for_backend_and_inference": True,
    "network_attempts_after_denial": network_attempts,
    "result": {
        "status": result.status.value,
        "requested_language": result.requested_language,
        "detected_language": result.detected_language,
        "transcript": result.transcript,
        "segments": [s.to_document() for s in result.segments],
        "model_id": result.model_id,
        "model_revision": result.model_revision,
        "runtime_name": result.runtime_name,
        "runtime_version": result.runtime_version,
    },
    "claims": {
        "accuracy": False,
        "wer": False,
        "clinical_quality": False,
        "production_authorization": False,
    },
    "pass": bool(
        actual_head == "c61370664488e4ad3e274ec61a233c46652ec6de"
        and missing_snapshot_pass
        and result.status.value == "success"
        and bool(result.transcript.strip())
        and network_attempts == []
    ),
}
out = Path("/content/cw006-colab-evidence.json")
out.write_text(json.dumps(evidence, ensure_ascii=False, indent=2), encoding="utf-8")
print("CW006_QUALIFICATION_RESULT=" + json.dumps(evidence, ensure_ascii=True, separators=(",", ":")))